# Week 12 Lab — Big Data Tools: SQL · APIs · PySpark
**Name:** ______________  **Student ID:** ______________  **Program:** ______________

Files needed next to this notebook: `week12_transactions.xlsx` (converted to CSV below) and `week12_api_sample.json`.

In [1]:
# Colab only: uncomment the next line FIRST (takes 2-3 minutes)
# !pip install pyspark

## Setup — convert the Excel file to CSV and build the SQLite database
Excel stores dates as serial numbers (days since 1899-12-30), so we convert them to real dates.

In [2]:
import pandas as pd
import sqlite3
import json

raw = pd.read_excel("week12_transactions.xlsx")
raw["date"] = pd.to_datetime(raw["date"], unit="D", origin="1899-12-30").dt.strftime("%Y-%m-%d")
raw.to_csv("week12_transactions.csv", index=False)

df = pd.read_csv("week12_transactions.csv")
conn = sqlite3.connect("momo.db")
df.to_sql("transactions", conn, if_exists="replace", index=False)
print(len(df), "rows loaded")
print("Total value:", df["amount_rwf"].sum(), "RWF")
df.head()

2000 rows loaded
Total value: 119876000 RWF


,txn_id,date,hour,district,txn_type,amount_rwf,flagged
0,T00001,2026-11-01,8,Gasabo,send,1000,0
1,T00002,2026-11-09,12,Nyarugenge,send,25000,0
2,T00003,2026-11-19,8,Muhanga,send,51000,0
3,T00004,2026-11-01,17,Huye,send,79000,0
4,T00005,2026-11-15,12,Huye,send,109000,0


# Part 1 · SQL with Python (12 marks)
## Exercise 1.1 — SELECT / WHERE / ORDER BY

In [3]:
# (a) 5 largest 'send' transactions above 100,000 RWF
q = """
SELECT txn_id, district, amount_rwf
FROM transactions
WHERE txn_type = 'send' AND amount_rwf > 100000
ORDER BY amount_rwf DESC, txn_id ASC
LIMIT 5
"""
pd.read_sql(q, conn)

,txn_id,district,amount_rwf
0,T01708,Kicukiro,140000
1,T00364,Huye,139000
2,T00895,Muhanga,139000
3,T00899,Nyarugenge,139000
4,T01314,Gasabo,139000


In [4]:
# (b) Count flagged transactions
q = "SELECT COUNT(*) AS flagged_count FROM transactions WHERE flagged = 1"
res = pd.read_sql(q, conn)
print(res)
assert res.loc[0, "flagged_count"] == 11   # checkpoint: 11

   flagged_count
0             11


**Note (a):** several transactions tie at 139,000 RWF, so I added `txn_id ASC` as a tie-breaker to make the top 5 reproducible.

## Exercise 1.2 — GROUP BY

In [5]:
# (a) Per txn_type: count, total, average — biggest total first
q = """
SELECT txn_type,
       COUNT(*)         AS n_txn,
       SUM(amount_rwf)  AS total_rwf,
       AVG(amount_rwf)  AS avg_rwf
FROM transactions
GROUP BY txn_type
ORDER BY total_rwf DESC
"""
by_type = pd.read_sql(q, conn)
print(by_type)
assert by_type.loc[0, "txn_type"] == "send" and by_type.loc[0, "total_rwf"] == 62049000

  txn_type  n_txn  total_rwf       avg_rwf
0     send    868   62049000  71485.023041
1  paybill    330   29970000  90818.181818
2  receive    489   27024000  55263.803681
3  airtime    313     833000   2661.341853


In [6]:
# (b) Total per district — which district leads?
q = """
SELECT district, SUM(amount_rwf) AS total_rwf
FROM transactions
GROUP BY district
ORDER BY total_rwf DESC
"""
by_district = pd.read_sql(q, conn)
print(by_district)
assert by_district.loc[0, "district"] == "Muhanga" and by_district.loc[0, "total_rwf"] == 21076000

     district  total_rwf
0     Muhanga   21076000
1      Gasabo   20738000
2        Huye   18736000
3  Nyarugenge   18091000
4    Kicukiro   11338000
5     Musanze   10490500
6      Rubavu   10110500
7   Nyagatare    9296000


**Answer:** `send` moves the most money (~62.0M RWF) and Muhanga leads the districts (~21.1M RWF).

## Exercise 1.3 — The busiest hour

In [7]:
# (a) SQL: transactions per hour, busiest first
q = """
SELECT hour, COUNT(*) AS n_txn
FROM transactions
GROUP BY hour
ORDER BY n_txn DESC
"""
hours_sql = pd.read_sql(q, conn)
print(hours_sql.head())

   hour  n_txn
0    20    185
1    17    183
2    18    176
3    12    170
4    11    169


In [8]:
# (b) The same question in Pandas
hours_pd = df["hour"].value_counts().sort_values(ascending=False)
print(hours_pd.head())

# Do both agree?
print("SQL busiest hour :", hours_sql.loc[0, "hour"], "with", hours_sql.loc[0, "n_txn"])
print("Pandas busiest   :", hours_pd.index[0], "with", hours_pd.iloc[0])
assert hours_sql.loc[0, "hour"] == hours_pd.index[0] == 20
assert hours_sql.loc[0, "n_txn"] == hours_pd.iloc[0] == 185   # checkpoint

hour
20    185
17    183
18    176
12    170
11    169
Name: count, dtype: int64
SQL busiest hour : 20 with 185
Pandas busiest   : 20 with 185


**Answer:** hour 20 (8 pm) is the busiest, with 185 transactions. SQL and Pandas agree.

# Part 2 · APIs & JSON (10 marks)
## Exercise 2.1 — JSON by hand

In [9]:
# NOTE: your notebook has its own JSON string - paste it here in place of this stand-in.
json_text = '{"city": "Kigali", "readings": [21.5, 23.0, 25.5, 24.0, 22.0]}'

data_obj = json.loads(json_text)        # text -> Python dictionary
city = data_obj["city"]
readings = data_obj["readings"]

print("(a) City:", city)
print("(b) Number of readings:", len(readings))
print("(c) Average:", round(sum(readings) / len(readings), 2))

(a) City: Kigali
(b) Number of readings: 5
(c) Average: 23.2


## Exercise 2.2 — Call a real API (Open-Meteo, Kigali)

In [10]:
import urllib.request

url = ("https://api.open-meteo.com/v1/forecast?latitude=-1.95&longitude=30.06"
       "&hourly=temperature_2m&forecast_days=1&timezone=Africa/Kigali")
try:
    with urllib.request.urlopen(url, timeout=10) as r:
        data = json.loads(r.read().decode())
    print("Live API worked")
except Exception as e:
    print("Offline - using sample file:", type(e).__name__)
    with open("week12_api_sample.json") as f:
        data = json.load(f)

Offline - using sample file: URLError


In [11]:
temps = data["hourly"]["temperature_2m"]
times = data["hourly"]["time"]

print("(a) Hours:", len(temps))
print("(b) Max:", max(temps), " Min:", min(temps))
hot_i = temps.index(max(temps))
print("(c) Hottest hour:", times[hot_i])

(a) Hours: 24
(b) Max: 27.6  Min: 15.0
(c) Hottest hour: 2026-11-24T14:00


## Exercise 2.3 — API data meets your toolbox

In [12]:
weather = pd.DataFrame({"time": times, "temp_c": temps})
print(weather["temp_c"].describe())          # (a)

mask = weather["temp_c"] >= 25               # (b) Boolean mask
print("Hours >= 25 C:", mask.sum())

count    24.000000
mean     20.387500
std       4.494375
min      15.000000
25%      16.050000
50%      19.450000
75%      24.475000
max      27.600000
Name: temp_c, dtype: float64
Hours >= 25 C: 6


# Part 3 · PySpark (10 marks)

In [13]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").appName("Week12").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

Picked up JAVA_TOOL_OPTIONS: -XX:+IgnoreUnrecognizedVMOptions -XX:-UseBASE64Intrinsics
Picked up JAVA_TOOL_OPTIONS: -XX:+IgnoreUnrecognizedVMOptions -XX:-UseBASE64Intrinsics


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/24 06:43:03 WARN Utils: Your hostname, vm, resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/11/24 06:43:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/usr/local/lib/python3.12/dist-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/11/24 06:43:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Exercise 3.1 — Load & inspect

In [14]:
sdf = spark.read.csv("week12_transactions.csv", header=True, inferSchema=True)
print("Rows:", sdf.count())
sdf.printSchema()
sdf.show(5)

Rows: 2000
root
 |-- txn_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- district: string (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- amount_rwf: integer (nullable = true)
 |-- flagged: integer (nullable = true)



+------+----------+----+----------+--------+----------+-------+
|txn_id|      date|hour|  district|txn_type|amount_rwf|flagged|
+------+----------+----+----------+--------+----------+-------+
|T00001|2026-11-01|   8|    Gasabo|    send|      1000|      0|
|T00002|2026-11-09|  12|Nyarugenge|    send|     25000|      0|
|T00003|2026-11-19|   8|   Muhanga|    send|     51000|      0|
|T00004|2026-11-01|  17|      Huye|    send|     79000|      0|
|T00005|2026-11-15|  12|      Huye|    send|    109000|      0|
+------+----------+----+----------+--------+----------+-------+
only showing top 5 rows


## Exercise 3.2 — Filter

In [15]:
n_big = sdf.filter(sdf.amount_rwf > 100000).count()
print("Transactions above 100,000 RWF:", n_big)
assert n_big == 451                           # checkpoint

sdf.filter(sdf.flagged == 1).show(5)

Transactions above 100,000 RWF: 451


+------+----------+----+--------+--------+----------+-------+
|txn_id|      date|hour|district|txn_type|amount_rwf|flagged|
+------+----------+----+--------+--------+----------+-------+
|T00279|2026-11-29|  19| Muhanga|    send|    127000|      1|
|T00326|2026-11-07|  22|    Huye|    send|    127000|      1|
|T00536|2026-11-27|  17|  Gasabo|    send|    127000|      1|
|T00655|2026-11-30|  17|  Rubavu|    send|    134000|      1|
|T00697|2026-11-01|   9| Muhanga|    send|    127000|      1|
+------+----------+----+--------+--------+----------+-------+
only showing top 5 rows


## Exercise 3.3 — groupBy

In [16]:
# (a) total per txn_type
sdf.groupBy("txn_type").sum("amount_rwf").show()

# (b) per-district totals -> Pandas, sorted
dist_spark = (sdf.groupBy("district").sum("amount_rwf").toPandas()
              .rename(columns={"sum(amount_rwf)": "total_rwf"})
              .sort_values("total_rwf", ascending=False))
print(dist_spark)

+--------+---------------+
|txn_type|sum(amount_rwf)|
+--------+---------------+
| paybill|       29970000|
| airtime|         833000|
| receive|       27024000|
|    send|       62049000|
+--------+---------------+



/usr/local/lib/python3.12/dist-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/dist-packages/pyspark/sql/pandas/conversion.py:374: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


     district  total_rwf
4     Muhanga   21076000
5      Gasabo   20738000
6        Huye   18736000
1  Nyarugenge   18091000
0    Kicukiro   11338000
2     Musanze   10490500
7      Rubavu   10110500
3   Nyagatare    9296000


In [17]:
# (c) Do Spark answers match SQL? They must.
# Yes: Spark send total = 62,049,000 and Muhanga = 21,076,000, identical to SQL in Exercise 1.2.
check = dist_spark.set_index("district")["total_rwf"].sort_index()
sql_check = by_district.set_index("district")["total_rwf"].sort_index()
print("Spark == SQL (districts):", (check == sql_check).all())

Spark == SQL (districts): True


## Bonus (+3) — Spark SQL

In [18]:
sdf.createOrReplaceTempView("transactions")
spark.sql("""
    SELECT txn_type, COUNT(*) AS n_txn, SUM(amount_rwf) AS total_rwf, AVG(amount_rwf) AS avg_rwf
    FROM transactions
    GROUP BY txn_type
    ORDER BY total_rwf DESC
""").show()

# Why is it powerful that the same SQL runs on SQLite and on a thousand-machine cluster?
# Because SQL is a standard language: you learn it once and reuse the same query unchanged
# on a laptop database or a huge cluster. The engine decides how to run it (Spark splits the
# work across machines), so analysts need not rewrite code when data grows.

+--------+-----+---------+------------------+
|txn_type|n_txn|total_rwf|           avg_rwf|
+--------+-----+---------+------------------+
|    send|  868| 62049000| 71485.02304147465|
| paybill|  330| 29970000| 90818.18181818182|
| receive|  489| 27024000|  55263.8036809816|
| airtime|  313|   833000|2661.3418530351437|
+--------+-----+---------+------------------+



# Part 4 · Choose Your Weapon (3 marks)
1. **SQL** — the data already sits in a PostgreSQL database, and monthly totals per district is a `GROUP BY` question that the database answers directly without copying data out.
2. **Pandas** — 40,000 rows easily fit in memory, and Pandas covers cleaning, charts and regression (with scikit-learn) in one tool.
3. **Spark** — 40 TB is far too big for one machine, and the data is already on a cluster, so Spark processes it in parallel where it lives.
4. **API** — a dashboard that needs live data with no file uploads must pull it from a weather service's API every time it loads.

# Part 5 · Final-Project Proposal (draft - adapt to your group)
**Question:** Can we predict whether a mobile-money transaction is above 100,000 RWF from its hour, district and transaction type? **Dataset:** the Week 12 mobile-money transactions dataset (November 2026, 2,000 rows, well above the 500-row minimum; 451 of them are above 100,000 RWF). **Why it matters:** high-value transactions drive most of the money moved, so knowing when and where they occur helps providers plan liquidity and fraud monitoring. **Model:** classification (logistic regression or decision tree), because the outcome is a yes/no label rather than a number; the model is easy to interpret and the two classes are reasonably balanced.